# Portfolio analysis of stock market indices with NumPy

In this project I analyze a portfolio split across four stock market indices (S&P 500, Nasdaq Composite, Dow Jones and Nikkei 225) using only NumPy, without pandas.

The question I answer is: how much did each index return and how much risk did it take over the last years, how much do they move together, what was the portfolio's worst loss from its peak, and what can be expected, within a realistic range, one year from now.

The data comes from FRED (Federal Reserve Bank of St. Louis) and the notebook downloads it by itself: the files are not included in the repository.

## 1. Downloading the data

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import requests

In [ ]:
# a single random generator for the whole notebook, with a fixed seed:
# this way the Monte Carlo simulation always gives the same results
rng = np.random.default_rng(seed=42)

In [ ]:
tickers = ["SP500", "NASDAQCOM", "DJIA", "NIKKEI225"]
folder_path = "/content/market_data"

# create the folder if it does not exist
os.makedirs(folder_path, exist_ok=True)

# loop over the tickers to download them
for ticker in tickers:
    # build the URL and the destination file path
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={ticker}"
    file_path = os.path.join(folder_path, f"{ticker}.csv")

    # perform the HTTP request
    response = requests.get(url, timeout=30)

    # check that the download succeeded (HTTP status code 200)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: {file_path}")
    else:
        print(f"Error downloading {ticker} (status code {response.status_code})")

## 2. Loading and cleaning

Before loading the data I describe the format of the FRED CSV files, because the parameters of `np.genfromtxt` depend on it.

Each file has two comma-separated columns. The first row is a text header, such as `observation_date,SP500`: the first column is the date, the second is named after the series code. Since it is text and not data, I skip it with `skip_header=1`.

Dates are in ISO format `YYYY-MM-DD` (for example `2020-03-16`), which NumPy reads directly as `datetime64[D]`. Prices are decimal numbers with a dot as the separator.

Missing values correspond to market holidays that still appear in the series: in those rows the price field does not contain a number but a dot `.` or is empty. With `missing_values="."` I tell `genfromtxt` to treat the dot as missing (an empty field already is by default), and with `filling_values=np.nan` I replace it with `NaN`.

In [ ]:
# function that loads a FRED CSV into a structured array
def load_series(path):
    # dates are read directly as datetime64[D], not as strings;
    # missing values become nan
    return np.genfromtxt(path, delimiter=",",
                         dtype=[("date", "datetime64[D]"), ("price", float)],
                         encoding="UTF-8",
                         skip_header=1,
                         missing_values=".",
                         filling_values=np.nan)


# call the function in a loop over the tickers and store the results in a dictionary
data = {}
for ticker in tickers:
    data[ticker] = load_series(os.path.join(folder_path, f"{ticker}.csv"))

In [ ]:
# number of rows and missing values in each series
for ticker in tickers:
    n_rows = data[ticker].shape[0]
    n_nan = np.sum(np.isnan(data[ticker]["price"]))
    print(f"{ticker}: {n_rows} rows, {n_nan} missing values")

Nasdaq and Nikkei have many more missing values than the other two because their series are much longer: on FRED the Nikkei starts in 1949. Each row of the structured array is a date-price pair, as the first rows of the Nikkei show.

In [ ]:
print(data["NIKKEI225"][:5])

## 3. Aligning the dates

The four series do not share the same dates: the Nikkei follows the Japanese market calendar, with holidays different from the American ones. To compare them I need only the days on which all four have a price.

I proceed like this: I remove the rows with a `nan` price from each series, find the common dates with `np.intersect1d` (which takes two arrays at a time, so I chain it), and then build boolean masks with `np.isin` to extract the prices on those dates.

In [ ]:
# remove the rows with a nan price from each series and store the clean series in a new dictionary
cleaned = {}
for ticker in tickers:
    nan_mask = ~np.isnan(data[ticker]["price"])
    cleaned[ticker] = data[ticker][nan_mask]

# check that no nan is left
for ticker in tickers:
    print(f"{ticker}: {np.sum(np.isnan(cleaned[ticker]['price']))} nan left")

In [ ]:
# start from the dates of the first series and intersect with the following series, one at a time
common_dates = cleaned[tickers[0]]["date"]
for ticker in tickers[1:]:
    common_dates = np.intersect1d(common_dates, cleaned[ticker]["date"])

print("Common dates:", common_dates.shape)

In [ ]:
# for each series, a mask that is True on the common dates
masks = {}
for ticker in tickers:
    masks[ticker] = np.isin(cleaned[ticker]["date"], common_dates)

# alignment check: the dates extracted from each series must match common_dates,
# otherwise the columns of the price matrix would be shifted without any error
for ticker in tickers:
    aligned = np.array_equal(cleaned[ticker]["date"][masks[ticker]], common_dates)
    print(f"{ticker} aligned: {aligned}")

In [ ]:
# extract the prices on the common dates, in the order of the tickers list, and stack them as columns:
# rows = days, columns = indices
price_columns = [cleaned[ticker]["price"][masks[ticker]] for ticker in tickers]
prices_matrix = np.column_stack(price_columns)

print("Dates:", common_dates.shape)     # (T,)
print("Prices:", prices_matrix.shape)   # (T, 4)

## 4. Returns and statistics per index

I now have a clean matrix with aligned dates, which I can work on with vectorized operations only. Returns have one row fewer than prices, because the first day has no previous day: that is why, when I need the date of a return, I use `common_dates[1:]`.

In [ ]:
# daily simple and log returns: shape (T-1, 4)
simple_returns = prices_matrix[1:] / prices_matrix[:-1] - 1
log_returns = np.log(prices_matrix[1:] / prices_matrix[:-1])

print(simple_returns.shape, log_returns.shape)

In [ ]:
# annualized volatility: standard deviation of daily log returns times the square root of 252,
# because the variance grows linearly with the number of days
annualized_volatility = np.std(log_returns, axis=0) * np.sqrt(252)

# 5th and 95th percentile of daily returns: shape (2, 4)
percentiles = np.percentile(log_returns, [5, 95], axis=0)

for i, ticker in enumerate(tickers):
    print(f"{ticker}: annual volatility {annualized_volatility[i]:.1%}, "
          f"5th percentile {percentiles[0, i]:.2%}, 95th percentile {percentiles[1, i]:.2%}")

In [ ]:
# worst day for each index: argmin along axis=0 looks for the minimum over time,
# separately for each column, and returns 4 positions
worst_idx_each = np.argmin(log_returns, axis=0)
worst_dates_each = common_dates[1:][worst_idx_each]
# paired fancy indexing: for each column, take the row of its minimum
worst_returns_each = log_returns[worst_idx_each, np.arange(len(tickers))]

for ticker, day, ret in zip(tickers, worst_dates_each, worst_returns_each):
    print(f"{ticker}: worst day {day} ({ret:.2%})")

In [ ]:
# worst day for the four markets together: mean of each day's returns (axis=1)
global_returns = np.mean(log_returns, axis=1)
worst_day_idx = np.argmin(global_returns)

print(f"Worst overall day: {common_dates[1:][worst_day_idx]} "
      f"(average drop {global_returns[worst_day_idx]:.2%})")

In [ ]:
# ratio between annualized return and volatility, and ranking from best to worst
annualized_return = np.mean(log_returns, axis=0) * 252
ratios = annualized_return / annualized_volatility

# argsort sorts in ascending order, so I reverse it
ranked_indices = np.argsort(ratios)[::-1]
ranked_tickers = np.array(tickers)[ranked_indices]

print("Ranking by return/volatility ratio:")
for ticker, score in zip(ranked_tickers, ratios[ranked_indices]):
    print(f"{ticker}: {score:.2f}")

## 5. Correlation

I compute the correlation matrix by hand: I standardize the returns with broadcasting (subtracting the mean and dividing by the standard deviation of each column) and then use a matrix product. When a matrix is standardized, its covariance equals its correlation.

For the result to match `np.corrcoef` I need to be consistent with the degrees of freedom (`ddof`). `np.std` uses `ddof=0` by default, dividing by $N$, so the product `Z.T @ Z` must also be divided by $N$. `np.corrcoef` internally works with $N - 1$, but that factor cancels out in the correlation formula, so the two matrices match. Mixing `ddof=0` in the standardization with $N - 1$ in the final division would instead give a different result.

In [ ]:
# standardization with broadcasting: (T-1, 4) minus (4,), divided by (4,)
standardized_returns = ((log_returns - np.mean(log_returns, axis=0))
                        / np.std(log_returns, axis=0))

# (4, T-1) @ (T-1, 4) -> (4, 4), divided by the number of observations
n_obs = standardized_returns.shape[0]
correlation_matrix = standardized_returns.T @ standardized_returns / n_obs

print(np.round(correlation_matrix, 2))
print("Matches np.corrcoef:",
      np.allclose(correlation_matrix, np.corrcoef(log_returns, rowvar=False)))

## 6. The portfolio

Weights: 40% S&P 500, 25% Nasdaq, 20% Dow Jones, 15% Nikkei 225.

To combine the indices I use simple returns, not log returns: the portfolio value is a weighted sum of the values of the single indices, so its return is the weighted average of the simple returns. Log returns add up nicely over time, but not across different assets.

Applying the same weights every day amounts to assuming that the portfolio is rebalanced daily to bring each index back to its share, with no transaction costs. In a buy-and-hold portfolio the weights would instead drift over time: the indices that rise the most would end up weighing more than their initial share.

In [ ]:
weights = np.array([0.40, 0.25, 0.20, 0.15])

# check: converting log returns back with expm1 gives the simple returns
print("expm1(log) matches the simple returns:",
      np.allclose(np.expm1(log_returns), simple_returns))

# daily portfolio return: (T-1, 4) @ (4,) -> (T-1,)
portfolio_returns = simple_returns @ weights

# cumulative value starting from 1: 1 + return is the growth factor of the day
cumulative_wealth = np.cumprod(1 + portfolio_returns)
print(f"The capital grew {cumulative_wealth[-1]:.2f} times")

In [ ]:
# running maximum up to each day, including the initial capital 1.0:
# without the comparison with 1.0, a loss on the first day would not be counted
running_max = np.maximum(1.0, np.maximum.accumulate(cumulative_wealth))

# daily drawdown: percentage drop from the peak reached up to that day
drawdowns = (cumulative_wealth - running_max) / running_max

max_dd_idx = np.argmin(drawdowns)
print(f"Maximum drawdown: {drawdowns[max_dd_idx]:.1%} on {common_dates[1:][max_dd_idx]}")

## 7. One-year Monte Carlo simulation

I simulate 10,000 scenarios of one trading year (252 days) with the bootstrap: each scenario is a sequence of historical portfolio returns drawn at random, with replacement. Replacement is needed because each simulated day must be able to draw any historical day, even one already drawn, and because the total number of draws (2.5 million) is far larger than the number of available returns.

Before creating the scenario matrix I estimate its memory: 10,000 × 252 float64 values of 8 bytes each is about 20 MB, so no problem.

In [ ]:
n_scenarios = 10_000
n_days = 252

estimated_mb = n_scenarios * n_days * 8 / 1e6
print(f"Estimated memory: {estimated_mb:.2f} MB")

# each row is a simulated year: 252 historical returns drawn at random with replacement
simulated_returns = rng.choice(portfolio_returns, size=(n_scenarios, n_days), replace=True)
print(f"Actual memory: {simulated_returns.nbytes / 1e6:.2f} MB")

In [ ]:
# compound the 252 growth factors of each scenario (axis=1): one final value per scenario
final_values = np.prod(1 + simulated_returns, axis=1)

mc_percentiles = np.percentile(final_values, [5, 50, 95])

# fraction of scenarios ending below the initial capital: the mean of a mask
prob_loss = np.mean(final_values < 1.0)

print(f"5th percentile (pessimistic scenario): {mc_percentiles[0]:.2f}")
print(f"Median: {mc_percentiles[1]:.2f}")
print(f"95th percentile (optimistic scenario): {mc_percentiles[2]:.2f}")
print(f"Probability of ending the year at a loss: {prob_loss:.1%}")

## 8. Charts

Portfolio value over time.

In [ ]:
# return dates start from the second common day
return_dates = common_dates[1:]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(return_dates, cumulative_wealth, color="tab:blue", linewidth=1.2)
# reference line: the initial capital
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Portfolio value (initial capital = 1)")
ax.set_xlabel("Date")
ax.set_ylabel("Multiple of initial capital")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Drawdown over time: how far the portfolio was below its peak, day by day.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
# area between 0 and the drawdown: the deeper it is, the further the portfolio is from its peak
ax.fill_between(return_dates, drawdowns * 100, 0, color="tab:red", alpha=0.4)
# mark the point of maximum drawdown
ax.scatter(return_dates[max_dd_idx], drawdowns[max_dd_idx] * 100, color="tab:red", zorder=3)
ax.set_title("Portfolio drawdown")
ax.set_xlabel("Date")
ax.set_ylabel("Distance from peak (%)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Distribution of the simulated final values, with the three percentiles.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(final_values, bins=80, color="tab:blue", alpha=0.7)
# one vertical line per percentile, with its label in the legend
labels = ["5th percentile", "Median", "95th percentile"]
styles = [":", "-", ":"]
for value, label, style in zip(mc_percentiles, labels, styles):
    ax.axvline(value, color="black", linestyle=style, linewidth=1.2,
               label=f"{label}: {value:.2f}")
# initial capital line: scenarios to its left end at a loss
ax.axvline(1.0, color="tab:red", linewidth=1.2, label="Initial capital")
ax.set_title("Portfolio value in one year: 10,000 simulated scenarios")
ax.set_xlabel("Multiple of initial capital")
ax.set_ylabel("Number of scenarios")
ax.legend()
fig.tight_layout()
plt.show()

## 9. Saving the results

I save the main results in an `.npz` archive, reload it and check that it matches the originals.

In [ ]:
results = {
    "dates": common_dates,
    "prices": prices_matrix,
    "log_returns": log_returns,
    "correlation": correlation_matrix,
    "mc_final_values": final_values,
    "mc_percentiles": mc_percentiles,
}
np.savez("portfolio_analysis.npz", **results)

loaded = np.load("portfolio_analysis.npz")
all_equal = all(np.array_equal(results[key], loaded[key]) for key in results)
print("Files:", loaded.files)
print("All arrays match the originals:", all_equal)

## 10. Summary

I analyzed the historical performance and risk profile of a global portfolio (40% S&P 500, 25% Nasdaq, 20% Dow Jones, 15% Nikkei 225).

Among the single indices, the Nasdaq had the best ratio between annualized return and volatility (0.76; this equals a Sharpe ratio only assuming a risk-free rate of zero), while the Dow Jones ranked last (0.63). Correlations confirm a strong co-movement among the US indices (above 0.81), while the Nikkei 225 offers partial diversification (correlation around 0.18). During global shocks, however, such as 16 March 2020 (average drop of 10.6% in one day), the benefits of diversification vanish.

Over the whole period the capital grew 3.97 times, with a maximum drawdown of 32.5% on 23 March 2020. The one-year Monte Carlo simulation gives a median value of 1.16 times the initial capital, 0.88 times in the pessimistic scenario (5th percentile) and 1.53 times in the optimistic one (95th percentile). The probability of ending the year at a loss is 19.4%. These figures should be read in light of the limitations below, and refer to data downloaded on 21 October 2026: re-running the notebook on another day changes them slightly.

Limitations:
- Nikkei time lag: the Tokyo exchange closes before New York opens, so aligning the series by date puts on the same row closing prices many hours apart. News from the US session reaches the Nikkei only the next day, so its same-day correlation with the US indices is probably underestimated.
- Favorable historical period: the Monte Carlo resamples only the returns of roughly the last ten years, a very favorable period for equities. The simulated scenarios inherit that high average return, so the median and the probability of loss are probably too optimistic.
- Different currencies: the Nikkei 225 is in yen, the other three indices in dollars. The portfolio combines returns in different currencies without converting them; for an investor based in euros they would all need to be converted, and currency risk would change both return and volatility.